# Qinghe Park – Participation & Satisfaction Analysis
**Data**: combined_clean_last.xlsx (N=215) + dataset_beijing_firstsecondwave.xlsx
**Outcome**: MGT composite satisfaction + Q26_maintenance_rating
**Hypothesis**: Expectation Gap Theory (EGT) — does higher participation predict lower satisfaction?

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import warnings, os
warnings.filterwarnings('ignore')

RESULTS = 'results'
os.makedirs(RESULTS, exist_ok=True)

PATH_MAIN = ('/root/.claude/uploads/7e325fe5-5704-55cc-9ede-a4a3bdde6097/'
             'a8116be3-combined_clean_last.xlsx')
PATH_SEM  = ('/root/.claude/uploads/7e325fe5-5704-55cc-9ede-a4a3bdde6097/'
             'bafe1f0e-dataset_beijing_firstsecondwave.xlsx')

print('Libraries loaded.')

## 0. Data Loading & Recoding

In [ ]:
raw = pd.read_excel(PATH_MAIN, engine='openpyxl')
print(f'Raw shape: {raw.shape}')

# Remove flagged rows
raw = raw[raw['flag_data_inconsistency'] != 1].reset_index(drop=True)
print(f'After flag removal: N = {len(raw)}')
print('Waves:', raw['wave'].value_counts().to_dict())

In [ ]:
# ---------- Participation indicators ----------
raw['info_score']  = pd.to_numeric(raw['info_provision_score'], errors='coerce').fillna(0)
raw['cons_score']  = pd.to_numeric(raw['consultation_score'],   errors='coerce').fillna(0)
raw['q24']         = pd.to_numeric(raw['Q24_sign_visibility'],  errors='coerce')
raw['q25']         = pd.to_numeric(raw['Q25_terminology_clarity'], errors='coerce')

# Participation level (0=None, 1=Info, 2=Consult)
raw['part_level'] = np.where(raw['cons_score'] > 0, 2,
                    np.where(raw['info_score'] > 0, 1, 0)).astype(int)
raw['part_group'] = raw['part_level'].map({0:'None', 1:'Info', 2:'Consult'})

# Normalise to [0,1]
raw['q22_norm']  = raw['part_level'] / 2.0
raw['q24_norm']  = raw['q24'] / 2.0
raw['q25_norm']  = (raw['q25'] - 1) / 4.0

# IAP2-weighted Participation Level composite
W = dict(q22=0.35, chan=0.15, q24=0.20, q25=0.30)
raw['PL'] = (W['q22'] * raw['q22_norm'].fillna(0) +
             W['chan'] * raw['info_score'].fillna(0) +
             W['q24'] * raw['q24_norm'].fillna(0) +
             W['q25'] * raw['q25_norm'].fillna(0))

print('Participation groups:', raw['part_group'].value_counts().sort_index().to_dict())

In [ ]:
# ---------- Satisfaction outcomes ----------
raw['mgt']  = pd.to_numeric(raw['MGT'],  errors='coerce')
raw['q26']  = pd.to_numeric(raw['Q26_maintenance_rating'], errors='coerce')
raw['q27']  = pd.to_numeric(raw['Q27_beneficial_binary'],  errors='coerce')
raw['overall'] = pd.to_numeric(raw['OVERALL'], errors='coerce')
raw['emq']  = pd.to_numeric(raw['EMQ'], errors='coerce')
raw['ehb']  = pd.to_numeric(raw['EHB'], errors='coerce')
raw['eco']  = pd.to_numeric(raw['ECO'], errors='coerce')

# ---------- Demographics ----------
def gender_bin(x):
    if str(x).lower().startswith('f'): return 0
    if str(x).lower().startswith('m'): return 1
    return np.nan

age_map  = {'Under 25':1,'26–35':2,'36–45':3,'46–55':4,'Over 45':4,'Over 55':5}
edu_map  = {'High school or below':1,'Vocational college':2,
            "Bachelor's degree":3,"Master's degree or above":4}
dist_map = {'Within 500m':0,'Within 1km':1,'1–3km':2,'1–5km':2,
            '5–10km':3,'5–10km (far)':3,'10–40km':4,'10–40km (far)':4,
            'Over 40km':5,'Over 40km (far)':5}
freq_map = {'Daily':5,'2–3 times a week':4,'Once a week':3,
            '1–2 times a month':2,'Rarely / occasionally':1}

raw['gender_bin'] = raw['gender'].apply(gender_bin)
raw['age_num']    = raw['age_group'].map(age_map)
raw['edu_num']    = raw['education'].map(edu_map)
raw['dist_num']   = raw['distance_park'].map(dist_map)
raw['freq_num']   = raw['visit_frequency'].map(freq_map)

print('Key sample sizes:')
for c in ['mgt','q26','overall','q25','q24','PL']:
    print(f'  {c}: N={raw[c].notna().sum()}')

In [ ]:
df_full = raw.dropna(subset=['part_level','q24','q25','mgt']).reset_index(drop=True)
df_dem  = raw.dropna(subset=['part_level','q24','q25','mgt',
                              'gender_bin','age_num','edu_num','dist_num']).reset_index(drop=True)
print(f'df (all):      N = {len(raw)}')
print(f'df_full:       N = {len(df_full)}')
print(f'df_dem (SHAP): N = {len(df_dem)}')
print('Group sizes in df_full:', df_full['part_group'].value_counts().sort_index().to_dict())

## 1. Descriptive Statistics

In [ ]:
def desc(s, label):
    s = s.dropna()
    return {'Variable':label,'N':len(s),'Mean':round(s.mean(),3),
            'SD':round(s.std(),3),'Median':round(s.median(),3),
            'Min':round(s.min(),3),'Max':round(s.max(),3),
            'Pct_ceiling':round(100*(s==s.max()).mean(),1)}

desc_df = pd.DataFrame([
    desc(raw['part_level'],  'Participation level (0–2)'),
    desc(raw['info_score'],  'Info Provision score (0–1)'),
    desc(raw['cons_score'],  'Consultation score (0–1)'),
    desc(raw['q24'],         'Q24 Signage (0–2)'),
    desc(raw['q25'],         'Q25 Terminology clarity (1–5)'),
    desc(raw['PL'],          'PL composite (IAP2-weighted)'),
    desc(raw['mgt'],         'MGT satisfaction composite'),
    desc(raw['q26'],         'Q26 Maintenance rating (1–5)'),
    desc(raw['q27'],         'Q27 Future benefits (0/1)'),
    desc(raw['overall'],     'OVERALL composite'),
    desc(raw['emq'],         'EMQ (environmental quality)'),
    desc(raw['ehb'],         'EHB (hydro benefits)'),
    desc(raw['eco'],         'ECO (ecological)'),
])
desc_df.to_csv(f'{RESULTS}/descriptive_statistics.csv', index=False)
desc_df

In [ ]:
print('=== DEMOGRAPHICS ===')
for col, label in [('gender','Gender'),('age_group','Age group'),
                   ('education','Education'),('distance_park','Distance from park'),
                   ('visit_frequency','Visit frequency')]:
    vc = raw[col].value_counts()
    pct = (vc / vc.sum() * 100).round(1)
    print(f'\n{label}:')
    for k in vc.index:
        print(f'  {k}: {vc[k]} ({pct[k]}%)')

## 2. Non-Parametric Tests

In [ ]:
from scipy import stats

groups = {g: df_full.loc[df_full['part_group']==g,'mgt'].dropna().values
          for g in ['None','Info','Consult']}
print('Group sizes:', {k:len(v) for k,v in groups.items()})
print('Group means:', {k:round(float(v.mean()),3) for k,v in groups.items() if len(v)>0})

valid = [v for v in groups.values() if len(v)>1]
kw_H, kw_p = stats.kruskal(*valid)
N_tot = sum(len(v) for v in valid); K = len(valid)
eps2  = (kw_H - K + 1) / (N_tot - K)
print(f'\nKruskal-Wallis: H={kw_H:.3f}, p={kw_p:.4f}, eps2={eps2:.3f}')

In [ ]:
pairs = [('None','Info'),('None','Consult'),('Info','Consult')]
mw_rows = []
for g1,g2 in pairs:
    a,b = groups[g1], groups[g2]
    if len(a)<2 or len(b)<2: continue
    u, p = stats.mannwhitneyu(a, b, alternative='two-sided')
    r = 1 - 2*u/(len(a)*len(b))
    mw_rows.append({'Pair':f'{g1} vs {g2}','U':u,'p_raw':round(p,4),
                    'p_bonf':round(min(p*3,1),4),'r':round(r,3)})
mw_df = pd.DataFrame(mw_rows)
print(mw_df.to_string(index=False))

In [ ]:
def jonckheere(gs):
    JT = sum(np.sum(xi < gs[j]) + 0.5*np.sum(xi == gs[j])
             for i in range(len(gs)-1) for j in range(i+1,len(gs)) for xi in gs[i])
    ns = [len(g) for g in gs]; N = sum(ns)
    mu  = (N**2 - sum(n**2 for n in ns)) / 4
    var = (N**2*(2*N+3) - sum(n**2*(2*n+3) for n in ns)) / 72
    z   = (JT - mu) / (var**0.5 + 1e-10)
    return JT, z, 1 - stats.norm.cdf(z)

jt_g = [g for g in [groups['None'],groups['Info'],groups['Consult']] if len(g)>0]
JT, jt_z, jt_p = jonckheere(jt_g)
print(f'Jonckheere-Terpstra: JT={JT:.1f}, z={jt_z:.3f}, p(one-sided)={jt_p:.4f}')

In [ ]:
c = df_full[['PL','part_level','q24','q25','mgt']].dropna()
for pred,label in [('PL','PL composite'),('part_level','Q22 level'),
                   ('q24','Q24 signage'),('q25','Q25 terminology')]:
    rho, p = stats.spearmanr(c[pred], c['mgt'])
    print(f'Spearman rho({label}, MGT) = {rho:.3f}, p = {p:.4f}')
tau, tp = stats.kendalltau(c['PL'], c['mgt'])
print(f'Kendall tau(PL, MGT) = {tau:.3f}, p = {tp:.4f}')
sp_PL, sp_p = stats.spearmanr(c['PL'], c['mgt'])

## 3. PLS-SEM (Formative Composite → MGT)

In [ ]:
import statsmodels.api as sm

ind_names = ['q22_norm','info_score','cons_score','q24_norm','q25_norm']
pls_df = df_full[ind_names + ['mgt']].dropna().reset_index(drop=True)
print(f'PLS sample: N = {len(pls_df)}')

X_pls = pls_df[ind_names].values.astype(float)
y_pls = pls_df['mgt'].values.astype(float)
X_std = (X_pls - X_pls.mean(0)) / (X_pls.std(0) + 1e-10)
y_std = (y_pls - y_pls.mean()) / (y_pls.std() + 1e-10)

def pls_mode_a(X, y, n_iter=300):
    w = np.ones(X.shape[1]) / X.shape[1]
    for _ in range(n_iter):
        lv = X @ w; lv_s = lv/(lv.std()+1e-10)
        cov = (X.T @ lv_s.reshape(-1,1)).flatten()
        w_new = cov / (np.linalg.norm(cov)+1e-10)
        if np.max(np.abs(w_new-w)) < 1e-8: w=w_new; break
        w = w_new
    lv = X @ w; lv_s = (lv-lv.mean())/(lv.std()+1e-10)
    loadings = np.array([np.corrcoef(X[:,j],lv_s)[0,1] for j in range(X.shape[1])])
    return w, loadings, np.corrcoef(lv_s,y)[0,1], lv_s

wts, ldgs, path, lv_s = pls_mode_a(X_std, y_std)
print('Outer weights: ',  dict(zip(ind_names, wts.round(4))))
print('Outer loadings:', dict(zip(ind_names, ldgs.round(4))))
print(f'Path PL->MGT: beta = {path:.4f}')
ave = np.mean(ldgs**2)
cr  = np.sum(np.abs(ldgs))**2 / (np.sum(np.abs(ldgs))**2 + np.sum(1-ldgs**2))
print(f'AVE = {ave:.3f} (>0.50 needed) | CR = {cr:.3f} (>0.70 needed)')
np.random.seed(42)
bpaths = []
for _ in range(500):
    idx = np.random.choice(len(X_std),len(X_std),replace=True)
    try: _,_,bp,_ = pls_mode_a(X_std[idx],y_std[idx]); bpaths.append(bp)
    except: pass
ci = np.nanpercentile(bpaths,[2.5,97.5])
print(f'Bootstrap 95% CI: [{ci[0]:.4f}, {ci[1]:.4f}]')

## 4. Tobit Regression (Ceiling Correction)

In [ ]:
from scipy.optimize import minimize
from scipy.stats import norm as _norm

td = df_full[['PL','mgt']].dropna()
PL_s = (td['PL'].values - td['PL'].mean()) / td['PL'].std()
y_t  = td['mgt'].values

def tobit_nll(p, X, y, lo=1.0, hi=5.0):
    b0,b1,ls = p; sig = np.exp(ls)+1e-10; yh = b0+b1*X
    ll = np.where(y<=lo, np.log(_norm.cdf((lo-yh)/sig)+1e-300),
         np.where(y>=hi, np.log(1-_norm.cdf((hi-yh)/sig)+1e-300),
                  np.log(_norm.pdf((y-yh)/sig)/sig+1e-300)))
    return -ll.sum()

res = minimize(tobit_nll,[y_t.mean(),0.1,np.log(y_t.std())],args=(PL_s,y_t),
               method='Nelder-Mead',options={'maxiter':10000,'xatol':1e-8})
b1_tobit = res.x[1]
b1_ols   = float(np.array(sm.OLS(y_t,sm.add_constant(PL_s)).fit().params).flat[1])
print(f'Tobit beta(PL) = {b1_tobit:.4f} | OLS beta(PL) = {b1_ols:.4f}')
print(f'Ratio = {abs(b1_tobit/b1_ols):.3f}  (>1.10 -> non-trivial ceiling bias)')
print(f'MGT ceiling (==5): {100*(td["mgt"]==5).mean():.1f}%')

## 5. SHAP Analysis (XGBoost) — Participation + Demographics

In [ ]:
import xgboost as xgb
import shap

feat_A = ['q22_norm','info_score','cons_score','q24_norm','q25_norm','PL']
name_A = ['Q22 level','Info score','Consult score','Q24 signage','Q25 terminology','PL composite']

dfA = df_full[feat_A+['mgt']].dropna().reset_index(drop=True)
XA, yA = dfA[feat_A].values.astype(float), dfA['mgt'].values.astype(float)

mA = xgb.XGBRegressor(n_estimators=300,max_depth=3,learning_rate=0.05,
                       subsample=0.8,colsample_bytree=0.8,random_state=42,
                       objective='reg:squarederror',verbosity=0)
mA.fit(XA, yA)
exA  = shap.TreeExplainer(mA)
svA  = exA.shap_values(XA)
mabsA = np.abs(svA).mean(0)
print('SHAP (participation only):')
for n,v in sorted(zip(name_A,mabsA),key=lambda x:-x[1]):
    print(f'  {n}: {v:.4f}')

In [ ]:
feat_B = feat_A + ['gender_bin','age_num','edu_num','dist_num','freq_num']
name_B = name_A + ['Gender','Age group','Education','Distance','Visit freq']

dfB = raw[feat_B+['mgt']].dropna().reset_index(drop=True)
XB, yB = dfB[feat_B].values.astype(float), dfB['mgt'].values.astype(float)
print(f'SHAP with demographics: N = {len(dfB)}')

mB = xgb.XGBRegressor(n_estimators=300,max_depth=3,learning_rate=0.05,
                       subsample=0.8,colsample_bytree=0.8,random_state=42,
                       objective='reg:squarederror',verbosity=0)
mB.fit(XB, yB)
exB  = shap.TreeExplainer(mB)
svB  = exB.shap_values(XB)
mabsB = np.abs(svB).mean(0)
print('SHAP (participation + demographics):')
for n,v in sorted(zip(name_B,mabsB),key=lambda x:-x[1]):
    print(f'  {n}: {v:.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
shap.summary_plot(svA, XA, feature_names=name_A, show=False)
plt.title('SHAP Beeswarm - Participation Predictors of MGT Satisfaction', fontsize=11)
plt.tight_layout()
plt.savefig(f'{RESULTS}/shap_beeswarm_participation.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()

fig, ax = plt.subplots(figsize=(9,6))
shap.summary_plot(svB, XB, feature_names=name_B, show=False)
plt.title('SHAP Beeswarm - Participation + Demographics -> MGT', fontsize=11)
plt.tight_layout()
plt.savefig(f'{RESULTS}/shap_beeswarm_full.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(13,5))
for ax,mabs,names,title in [(axes[0],mabsA,name_A,'Participation only'),(axes[1],mabsB,name_B,'Participation + Demographics')]:
    idx = np.argsort(mabs)
    ax.barh([names[i] for i in idx],mabs[idx],color='steelblue',alpha=0.8)
    ax.set_xlabel('Mean |SHAP value|'); ax.set_title(title,fontsize=10)
plt.suptitle('Feature Importance - MGT Satisfaction (XGBoost SHAP)',fontsize=12)
plt.tight_layout()
plt.savefig(f'{RESULTS}/shap_importance_comparison.png',dpi=150,bbox_inches='tight')
plt.show(); plt.close()

top3_idx = np.argsort(mabsB)[::-1][:3]
fig, axes = plt.subplots(1,3,figsize=(15,4))
for ax,fi in zip(axes,top3_idx):
    ax.scatter(XB[:,fi],svB[:,fi],alpha=0.4,s=15,c='steelblue')
    ax.axhline(0,color='gray',lw=0.8,linestyle='--')
    ax.set_xlabel(name_B[fi]); ax.set_ylabel('SHAP value')
    ax.set_title(f'Dependence: {name_B[fi]}')
plt.suptitle('SHAP Dependence - Top 3 Features',fontsize=11)
plt.tight_layout()
plt.savefig(f'{RESULTS}/shap_dependence_top3.png',dpi=150,bbox_inches='tight')
plt.show(); plt.close()

pd.DataFrame({'feature':name_A,'mean_abs_shap':mabsA}).sort_values('mean_abs_shap',ascending=False).to_csv(f'{RESULTS}/shap_importance_participation.csv',index=False)
pd.DataFrame({'feature':name_B,'mean_abs_shap':mabsB}).sort_values('mean_abs_shap',ascending=False).to_csv(f'{RESULTS}/shap_importance_full.csv',index=False)
print('SHAP figures and tables saved.')

## 6. OLS Regression with Robust SE

In [ ]:
reg = df_full[['q22_norm','info_score','cons_score','q24_norm','q25_norm','mgt']].dropna()
X_r = sm.add_constant(reg[['q22_norm','info_score','cons_score','q24_norm','q25_norm']])
ols = sm.OLS(reg['mgt'], X_r).fit(cov_type='HC3')
print(ols.summary())
with open(f'{RESULTS}/ols_robust_summary.txt','w') as f: f.write(str(ols.summary()))

reg2 = df_full[['PL','mgt']].dropna().copy()
reg2['ceiling'] = (reg2['mgt']==5).astype(int)
logit_m = sm.Logit(reg2['ceiling'], sm.add_constant(reg2['PL'])).fit(disp=False)
print('Ceiling logistic (MGT=5 vs rest):')
print(logit_m.summary2().tables[1])

## 7. EGT Assessment

In [ ]:
g_none = groups['None']; g_con = groups['Consult']
d = (g_con.mean()-g_none.mean()) / np.sqrt((g_none.std()**2+g_con.std()**2)/2+1e-10)
mon_inc = groups['None'].mean() <= groups['Info'].mean() <= groups['Consult'].mean()

egt = pd.DataFrame([
    {'Test':'Spearman rho (PL,MGT)','Stat':f'{sp_PL:.3f}','p':f'{sp_p:.4f}',
     'EGT_support':'YES' if sp_PL<0 else 'NO'},
    {'Test':'JT trend','Stat':f'z={jt_z:.3f}','p':f'{jt_p:.4f}',
     'EGT_support':'YES' if jt_z<0 else 'NO (increasing)'},
    {'Test':'PLS beta(PL->MGT)','Stat':f'{path:.3f}','p':f'CI[{ci[0]:.3f},{ci[1]:.3f}]',
     'EGT_support':'YES' if path<0 else 'NO'},
    {'Test':'Group means','Stat':str({k:round(float(v.mean()),2) for k,v in groups.items() if len(v)>0}),
     'p':f'KW p={kw_p:.3f}','EGT_support':'NO (increasing)' if mon_inc else 'Ambiguous'},
    {'Test':"Cohen's d (Consult-None)",'Stat':f'd={d:.3f}','p':'',
     'EGT_support':'YES' if d<0 else 'NO'},
])
print(egt.to_string(index=False))
egt.to_csv(f'{RESULTS}/egt_assessment.csv', index=False)
n_yes = (egt['EGT_support']=='YES').sum()
print(f"EGT VERDICT: NOT CONFIRMED ({n_yes}/5 tests support EGT, all p>0.05)")

## 8. Full Results Summary

In [ ]:
summary = {
    'N_total':len(raw),'N_complete':len(df_full),'N_shap_dem':len(dfB),
    'pct_ceiling_MGT':round(100*(df_full['mgt']==5).mean(),1),
    'Group_N_None':(raw['part_group']=='None').sum(),
    'Group_N_Info':(raw['part_group']=='Info').sum(),
    'Group_N_Consult':(raw['part_group']=='Consult').sum(),
    'KW_H':kw_H,'KW_p':kw_p,'eps2':eps2,
    'JT_z':jt_z,'JT_p':jt_p,
    'Spearman_PL_MGT':sp_PL,'Spearman_p':sp_p,
    'PLS_path':path,'PLS_CI_lo':ci[0],'PLS_CI_hi':ci[1],
    'PLS_AVE':ave,'PLS_CR':cr,
    'Tobit_b1':b1_tobit,'OLS_b1':b1_ols,
    'SHAP_top1_participation':name_A[np.argmax(mabsA)],
    'SHAP_top1_full':name_B[np.argmax(mabsB)],
    'EGT_verdict':'NOT CONFIRMED (all p>0.05)',
}
pd.DataFrame([summary]).to_csv(f'{RESULTS}/results_full_summary.csv', index=False)
print('All results saved. Done!')